# Overlap and divergence of anomaly scores

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
se_cols = [
    'mse',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_97',
    # 'mse_95',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

se_rank_cols = [
    f"rank_{col}" for col in se_cols
]

# ----------------------------------------------
rse_cols = [
    f"{col}_rel" for col in se_cols
]

rse_rank_cols = [
    f"rank_{col}_rel" for col in se_cols
]
# ----------------------------------------------
se_family = [
    'mse',
    'mse_97',
    # 'mse_95',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

rse_family = [
    f'{col}_rel' for col in se_family
]

# Custom functions

## IDs top anomalies

In [4]:
def get_ids_set(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [5]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    ids_top_list = []

    for score in scores_list:

        ids_set = get_ids_set(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

        ids_top_dict[score] = ids_set

        ids_top_list += list(ids_set)


    n_dictinct_top = len(set(ids_top_list))

    print(f"N unique: {n_dictinct_top}")


    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_dictinct_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.4f}%")

    return unique_ids_dict, ids_top_dict, n_dictinct_top


In [6]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}
    ids_top_list = []


    for score in scores_list:

        ids_set = get_ids_set(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

        ids_top_dict[score] = ids_set

        ids_top_list += list(ids_set)

    n_dictinct_top = len(set(ids_top_list))
    print(f"N unique: {n_dictinct_top}")

    common_ids_set = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_set)
    common_pct = n_common/n_dictinct_top*100 
    print(f"N common:\n{n_common} -- > {common_pct:4f}%")

    return common_ids_set, ids_top_dict, n_dictinct_top, n_common 

# Config

## Directories

In [7]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_id = 'bin_03'
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"
os.makedirs(f"{ch_4_dir}/{bin_id}", exist_ok=True)

## Data

In [8]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

## scores

In [9]:
score_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)

rank = np.arange(score_df.shape[0]) + 1
score_rank_df = score_df.copy()
# score_rank_df
for col in score_df.columns:

    index_sorted = score_df.sort_values(
        by=col, ascending=False
    ).index

    score_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_rank_df[f'rank_{col}'].astype(int)

n_spec = score_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [10]:
score = 'mse_97'
score_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(5)

,mse_97,rank_mse_97
specobjid,,
1919783100783552512,14.699505,1.0
2245159102839810048,13.236768,2.0
2930814289679771648,10.930245,3.0
1071977423131666432,10.445363,4.0
969534273977608192,9.671392,5.0


# IDs per score
Get specobjid for top 1\% of anomalies of each score

In [11]:
ids_top_dict = {}

all_scores = se_cols + rse_cols

for score in all_scores:

    ids_top_dict[score] = get_ids_set(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

In [12]:
[
    n for n 
    in [len(v) for v in ids_top_dict.values()]
]

[1819, 1819, 1819, 1819, 1819, 1819, 1819, 1819]

# All scores

## Common anomalies

In [13]:
all_scores = se_cols + rse_cols 
all_common_ids, all_ids_top_dict, n_dictinct_all, n_common_all = top_common_ids(
    scores_df=score_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 4095
N common:
467 -- > 11.404151%


In [15]:
len(all_common_ids)

467

In [14]:
spec_idx_common_list = []

for objid in all_common_ids:

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    spec_idx_common_list.append(spec_idx)
len(spec_idx_common_list)

467

In [ ]:
# selection of common anomalies
common_selected_ids = [
    1413149843194406912, # spike in blue end
    
]

In [41]:
all_common_ids_list = list(all_common_ids)
rank_cols = se_rank_cols + rse_rank_cols
score_rank_df.loc[
    # all_common_ids_list,
    [
        # spike and noise
        1413149843194406912, 637325355518027776,
        # 
        734111514142730240, # strong narrow emission line
        # braod emssion line with OIII way larger than H alpha
        1633733043925575680,
        # broad with weird continuum, and missing flux on red end and OIII in half 
        1192355533059811328,
        # weird continuum
        531492683672217600,
        # weird blue slope, very pronounced
        1959124163192973312,
        # star?
        1415343918345644032, 2506362108355045376,
        1780176998165932032, # this has to go
    ],
    rank_cols
]

,rank_mse,rank_mse_filter_250,rank_mse_97,rank_mse_filter_250_97,rank_mse_rel,rank_mse_filter_250_rel,rank_mse_97_rel,rank_mse_filter_250_97_rel
specobjid,,,,,,,,
1413149843194406912,2.0,2.0,52.0,37.0,49.0,17.0,31.0,30.0
637325355518027776,190.0,86.0,27.0,21.0,56.0,15.0,20.0,18.0
734111514142730240,14.0,128.0,63.0,83.0,4.0,267.0,118.0,155.0
1633733043925575680,73.0,118.0,22.0,25.0,93.0,88.0,18.0,20.0
1192355533059811328,110.0,237.0,80.0,115.0,129.0,259.0,92.0,117.0
531492683672217600,293.0,135.0,11.0,11.0,83.0,35.0,8.0,7.0
1959124163192973312,285.0,277.0,42.0,39.0,508.0,563.0,114.0,130.0
1415343918345644032,588.0,239.0,70.0,59.0,209.0,115.0,38.0,27.0
2506362108355045376,655.0,417.0,98.0,94.0,253.0,230.0,47.0,51.0


## Distinct anomalies

In [32]:
all_scores = se_cols + rse_cols

all_unique_ids_dict, all_ids_top_dict, n_dictinct_top_all = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=all_scores,
    quantile=99,
    # n_top=1000, use_ntop=True
)

N unique: 4095
Unique to mse:
262 --> 6.3980%
Unique to mse_filter_250:
128 --> 3.1258%
Unique to mse_97:
46 --> 1.1233%
Unique to mse_filter_250_97:
61 --> 1.4896%
Unique to mse_rel:
45 --> 1.0989%
Unique to mse_filter_250_rel:
125 --> 3.0525%
Unique to mse_97_rel:
64 --> 1.5629%
Unique to mse_filter_250_97_rel:
146 --> 3.5653%


In [33]:
ensemble_diversity =  n_dictinct_top_all/(1819*8)*100
print(f"Ensemble diversity: {ensemble_diversity:.2f}%")
# --------------------------------------------------------------------
n_unique_per_score_all_all = 262 + 128 + 46 + 61 + 45 + 125 + 64 + 146
common_over_distinct = n_unique_per_score_all_all/n_dictinct_top_all*100
print(f"Common over distinct: {common_over_distinct:.2f}%")

Ensemble diversity: 28.14%
Common over distinct: 21.42%


# old code

# Distinct IDs

## SE family

In [14]:
se_unique_ids_dict, se_ids_top_dict, n_dictinct_top_se = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3173
Unique to mse:
433 --> 13.6464%
Unique to mse_filter_250:
261 --> 8.2257%
Unique to mse_97:
63 --> 1.9855%
Unique to mse_filter_250_97:
143 --> 4.5068%


In [15]:
n_dictinct_top_se/(1819*4)*100

43.60912589334799

In [16]:
n_unique_per_score_all = 433 + 261 + 63 + 143
n_unique_per_score_all/n_dictinct_top_se*100

28.364323983611722

## RSE family

In [17]:
rse_unique_ids_dict, rse_ids_top_dict, n_dictinct_top_rse = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=rse_cols,
    quantile=99,
    # n_top=1000, use_ntop=True
)

N unique: 3107
Unique to mse_rel:
309 --> 9.9453%
Unique to mse_filter_250_rel:
190 --> 6.1152%
Unique to mse_97_rel:
158 --> 5.0853%
Unique to mse_filter_250_97_rel:
177 --> 5.6968%


In [18]:
n_dictinct_top_rse/(1819*4)*100

42.702034084661896

In [19]:
n_unique_per_score_all_rse = 309 + 190 + 158 + 177
n_unique_per_score_all_rse/n_dictinct_top_rse*100

26.842613453492113

# Common IDs

## SE family

In [23]:
se_common_ids, se_ids_top_dict, n_dictinct_se, n_common_se = top_common_ids(
    scores_df=score_df.copy(), scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3173
N common:
692 -- > 21.809014%


In [24]:
n_dictinct_se, n_common_se

(3173, 692)

## RSE Family

In [26]:
rse_common_ids, res_ids_top_dict, n_dictinct_rse, n_common_rse = top_common_ids(
    scores_df=score_df.copy(), scores_list=rse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N unique: 3107
N common:
789 -- > 25.394271%


In [27]:
n_dictinct_rse, n_common_rse

(3107, 789)